In [0]:
from pyspark.sql.types import StructField,StringType
from pyspark.sql.functions import *

path = '/Volumes/dev001_db_wp/standard/student_demo/currupted.txt'

# fetch schema and store in a variable
schm = spark.read.format('csv').options(header='true',inferSchema='true').load(path).schema
new_schm = schm.add(StructField('corrupt_data',StringType(),True))

df = spark.read.format('csv').schema(new_schm).options(header='true',mode='PERMISSIVE',ColumnNameOfCorruptRecord='corrupt_data').load(path)
# print(new_schm)

df.cache()

corrupt_df = df.select('corrupt_data').filter(col('corrupt_data').isNotNull())
display(corrupt_df)

corrupt_data
"4rohit,40000,pune"
"7,ajaymumbai,chennai"
"8,parth"


In [0]:
# Create 2 dataframes
df1= spark.createDataFrame([(1,'A'),(2,'B'),(3,'X')],['id','name'])
df2= spark.createDataFrame([(3,'X'),(4,'Y'),(1,'A')],['id','name'])

# cross join
cross_joined_df = df1.crossJoin(df2)

# Anti join (returns rows present in df1 but not in df2)
anti_joined_df = df1.join(df2, df1.id == df2.id, 'anti')

display(df1)
display(df2)

display(anti_joined_df)

id,name
1,A
2,B
3,X


id,name
3,X
4,Y
1,A


id,name
2,B


In [0]:
emp_df = spark.read.format('csv').options(header='true',inferSchema='true').load('/Volumes/dev001_db_wp/standard/student_demo/employees.csv')

dept_df = spark.read.format('csv').options(header='true',inferSchema='true').load('/Volumes/dev001_db_wp/standard/student_demo/department.csv')

# Sort Merge Join
# Both the datasets are sorted based on join key. Then , the sorted datasets are merge/join
# This join is efficient for large datasets are sorted on the join key --> because it will perform streamlined comparison and that's why this is faster .
# Dataframes are sorted on the join key

sorted_emp_id = emp_df.orderBy('DEPARTMENT_ID')
sorted_dept_df = dept_df.orderBy('ID')

# spark will perform a Sort Merge Join

joined_df = sorted_emp_id.join(sorted_dept_df, sorted_emp_id.DEPARTMENT_ID == sorted_dept_df.ID,'inner')

# display(sorted_emp_id)
# display(sorted_dept_df)

display(joined_df)

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,ID,Department_Name
198,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,50,50,Sales
199,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,50,50,Sales
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10,10,IT
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10,10,IT
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20,20,HR
202,Pat,null,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20,20,HR
203,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,null,-,101,40,40,Marketing
204,Hermann,Baer,HBAER,515.123.8888,null,PR_REP,10000,-,101,70,70,Accounts
100,Steven,null,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90,90,Research
101,Neena,Kochhar,NKOCHHAR,515.123.4568,21-Sep-05,AD_VP,17000,-,100,90,90,Research


In [0]:
# set braodcast size threshold
# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "20000000")

# disable automatic broadcast join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [0]:
from pyspark.sql.functions import *

# large df --> 1 gb
emp_large_df = spark.read.format('csv').options(header='true',inferSchema='true').load('/Volumes/dev001_db_wp/standard/student_demo/employees.csv')

# small df --> 10 mb
dept_small_df = spark.read.format('csv').options(header='true',inferSchema='true').load('/Volumes/dev001_db_wp/standard/student_demo/department.csv')

# Perform manual broadcast join
df_join = emp_large_df.join(broadcast(dept_small_df), emp_large_df.DEPARTMENT_ID == dept_small_df.ID)
display(df_join)


EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID,ID,Department_Name
198,Donald,OConnell,DOCONNEL,650.507.9833,21-Jun-07,SH_CLERK,2600,-,124,50,50,Sales
199,Douglas,Grant,DGRANT,650.507.9844,13-Jan-08,SH_CLERK,2600,-,124,50,50,Sales
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10,10,IT
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10,10,IT
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20,20,HR
202,Pat,null,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20,20,HR
203,Susan,Mavris,SMAVRIS,515.123.7777,07-Jun-02,HR_REP,null,-,101,40,40,Marketing
204,Hermann,Baer,HBAER,515.123.8888,null,PR_REP,10000,-,101,70,70,Accounts
100,Steven,null,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90,90,Research
101,Neena,Kochhar,NKOCHHAR,515.123.4568,21-Sep-05,AD_VP,17000,-,100,90,90,Research


In [0]:
from pyspark.sql.functions import *
from pyspark import StorageLevel

# 1gb data but memory 800mb
emp_df = spark.read.format('csv').options(header='true',inferSchema='true').load('/Volumes/dev001_db_wp/standard/student_demo/employees.csv')

# Cache the Dataframe
emp_df.cache()

# Persist the Dataframe in memory
# emp_df.persist(StorageLevel.MEMORY_ONLY) # This is equal to df.cache()

# Persist the Dataframe in memory and disk
# emp_df.persist(StorageLevel.MEMORY_AND_DISK)

# Persist the Dataframe in Disk only
# emp_df.persist(StorageLevel.DISK_ONLY)

# 1st transformation
res = emp_df.filter(col('SALARY')>3000)
display(res)

# Check data stored in memory or not
print(emp_df.storageLevel.useMemory)

# unpersist() --> It removes the Dataframe from memory and or disk if it was previously cached or persisted. It frees up the memory

# emp_df.unpersist()

# print(emp_df.storageLevel.useMemory)


res.cache()

# # 2nd transformation --> job id wise salary
total_salary_df = res.groupBy('JOB_ID').agg(sum('SALARY').alias('Total_Salary'))
display(total_salary_df)

# 3rd transsformation is on res.

# 4th transformation is on res

EMPLOYEE_ID,FIRST_NAME,LAST_NAME,EMAIL,PHONE_NUMBER,HIRE_DATE,JOB_ID,SALARY,COMMISSION_PCT,MANAGER_ID,DEPARTMENT_ID
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10
200,Jennifer,Whalen,JWHALEN,515.123.4444,17-Sep-03,AD_ASST,4400,-,101,10
201,Michael,Hartstein,MHARTSTE,515.123.5555,17-Feb-04,MK_MAN,13000,-,100,20
202,Pat,null,PFAY,603.123.6666,17-Aug-05,MK_REP,6000,-,201,20
205,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110
204,Hermann,Baer,HBAER,515.123.8888,null,PR_REP,10000,-,101,70
205,Shelley,Higgins,SHIGGINS,515.123.8080,07-Jun-02,AC_MGR,12008,-,101,110
206,William,Gietz,WGIETZ,515.123.8181,07-Jun-02,AC_ACCOUNT,8300,-,205,110
100,Steven,null,SKING,515.123.4567,17-Jun-03,AD_PRES,24000,-,-,90
101,Neena,Kochhar,NKOCHHAR,515.123.4568,21-Sep-05,AD_VP,17000,-,100,90


True


JOB_ID,Total_Salary
FI_ACCOUNT,30600
MK_MAN,13000
IT_PROG,28800
FI_MGR,12008
AC_ACCOUNT,8300
PU_CLERK,3100
AC_MGR,24016
PR_REP,10000
ST_MAN,36400
MK_REP,6000


In [0]:
# 3 Avoid wide transformation
# 4 Avoid inferschema
emp_df = spark.read.format('csv').options(header='true').load('/Volumes/dev001_db_wp/standard/student_demo/employees.csv')

In [0]:
emp_df = spark.read.format('csv').options(header='true').load('/Volumes/dev001_db_wp/standard/student_demo/10000Records.csv')

res = emp_df.repartition(3)

# res = emp_df.filter(col('Salary')>20000)
# display(res)

# res.coalesce(1).write.mode('overwrite').format('csv').option('header','true').save('/Volumes/dev001_db_wp/standard/student_demo/output/my_data')

res.repartition(2).write.mode('overwrite').format('csv').option('header','true').save('/Volumes/dev001_db_wp/standard/student_demo/output/my_data')


In [0]:
# emp_df = spark.read.format('csv').options(header='true').load('/Volumes/dev001_db_wp/standard/student_demo/10000Records.csv')

# print(emp_df.rdd.getNumPartitions())
# # display(emp_df)

---------------------------------------------------------------------------
PySparkNotImplementedError                Traceback (most recent call last)
File <command-7857006970415175>, line 3
      1 emp_df = spark.read.format('csv').options(header='true').load('/Volumes/dev001_db_wp/standard/student_demo/10000Records.csv')
----> 3 print(emp_df.rdd.getNumPartitions())

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:265, in _wrap_property.<locals>.wrapper(self)
    263 start = time.perf_counter()
    264 try:
--> 265     res = prop.fget(self)
    266     logging_helper.log_event(
    267         accessor=wrapper,
    268         module_name=module_name,
   (...)
    271         duration=time.perf_counter() - start
    272     )
    273     return res

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:2432, in DataFrame.rdd(self)
   2430 @property
   2431 def rdd(self) -> "RDD[Row]":
-> 2432     raise PySparkNotImplementedError(
   

### **Broadcast variable** 
-Brodcast variable in spark is a way to efficiently share small piece of data (like a lookup table or constant) across all worker nodes in a spark cluster. 

-spark send this broadcast variable to each node. Each worker node keeps the copy of the broadcast variable  and can use it during its transformation.
Its value can be accessed by calling value method

In [0]:
from pyspark.sql.functions import *
emp_df = spark.read.format('csv').options(header='true').load('/content/employees.csv')

# Small dicationary that maps dept id with dept name
Dept_Names = {'10':'IT','20':'SALES', '30':'PRODUCTION','40':'CHEMICAL','50':'FINANCE','60':'OPERATION','70':'HR'} 

# Broadcast the Dept_Names disctionary as a variable
deptname_broadcasted = spark.sparkContext.broadcast(Dept_Names)
# print(deptname_broadcasted.value)

# function to fetch department name based on department id using the broadcast variable
def fetch_dept_name(dept_id):
  dept_name = deptname_broadcasted.value.get(str(dept_id),'Unknown')
  return dept_name

# Register the Python function as a UDF (user defined function)/ spark function
sparkf_fetch_dept_name = udf(fetch_dept_name)


#Apply the UDF to fetch department name based on department id
final_df = emp_df.withColumn('Department_Name',sparkf_fetch_dept_name(col('DEPARTMENT_ID')))
# display(emp_df)
final_df.show()

---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-8192602629158684>, line 7
      4 Dept_Names = {'10':'IT','20':'SALES', '30':'PRODUCTION','40':'CHEMICAL','50':'FINANCE','60':'OPERATION','70':'HR'} 
      6 # Broadcast the Dept_Names disctionary as a variable
----> 7 deptname_broadcasted = spark.sparkContext.broadcast(Dept_Names)
      8 print(deptname_broadcasted)

File /databricks/spark/python/pyspark/sql/connect/session.py:967, in SparkSession.__getattr__(self, name)
    965 def __getattr__(self, name: str) -> Any:
    966     if name in ["_jsc", "_jconf", "_jvm", "_jsparkSession", "sparkContext", "newSession"]:
--> 967         raise PySparkAttributeError(
    968             error_class="JVM_ATTRIBUTE_NOT_SUPPORTED", message_parameters={"attr_name": name}
    969         )
    970     return object.__getattribute__(self, name)

PySparkAttributeError: [JVM_ATTRIBUTE_N

### **Accumulator** 
- An Accumulator is a spark variable used to aggragate or accumulate information across tasks runnning in a spark job. They are primarily used for counters or aggragating numerical value.

In [0]:
from pyspark.sql.functions import *

# Sample employ data with some negative salaries
data = [(1, 'Alice',3000),(2,'Bob',-1000),(3,'Charlie',2500),(4,'David',-500),(5,'Vishal',-799)]

df = spark.createDataFrame(data, schema= ['emp_id','name','salary'])

# define an accumulator for counting invalid salaries
invalid_salary_acc = spark.sparkContext.accumulator(0)

# function to check for invalid salaries and update the accumulator
def check_invalid_salary(salary):
  if salary<0:
    invalid_salary_acc.add(1) # increment the accumulator if salary is invalid
  return salary   

# Register the function as a UDF
sparkf_check_invalid_salary = udf(check_invalid_salary)

# apply the UDF to check invalid salary
df_checked = df.withColumn('checked_salary',sparkf_check_invalid_salary(col('salary')))

df_checked.show()

# After action, the accumulator holds the count of invalid salaries
print(f"Number of invalid salaries",invalid_salary_acc.value)
# df.show()


In [0]:
spark-submit --master yarn --number-of-worker 10 --number-of-executor 30 --executor-memory 8gb --core 88 --deploy-mode client my_spark_script.py

In [0]:
spark-submit --master yarn --deploy-mode cluster my_spark_script.py

In [0]:
# i want to process 50 GB
# cluster size --> number of workers? executor ? executor memory?

# double the cluste configuration--. 2 times ---> 100GB 
Total Memory target = 100GB

1)Decide executor memory --> common range per executor 4-16 GB --> lets choose 8GB per executor memory

2)calculate number of executors = Total memory target/executor memory = 100/8 = 13 executor

3)Decide number of cores per executor --> each executor should ideally have 2-4 cores for efficient processing.
Lets say you choose 4 cores per executor.

4)Calculate total cores = number of cores per executor * number of executors = 4 * 13 = 52 cores

5)
Ideal cores per node: 16 cores (production)
Number of nodes = total cores/ cores per node =  52 / 16 = 3.25 ≈ 4 nodes


In [0]:
it generates 2 plans
1) Logical plan - original query structure
2) Physical plan -convert this logical plan into 1 or more physical plans and selects the most efficient one based on the cost
after selecting 1 phyical plan based on cost it applies predicate pushdown